In [ ]:
%sql
/* 
Purpose: Comprehensive Test Code for f_invntry_bal_dly_hist Table in Databricks Environment
*/

/* 
Section: Setup and Configuration 
Description: Set up the testing environment by ensuring necessary tables exist and are accessible.
*/

-- Ensure the f_invntry_bal_dly_hist table exists
CREATE OR REPLACE TABLE purgo_playground.f_invntry_bal_dly_hist (
    co_key STRING,
    g_account STRING,
    g_assignment_type STRING,
    g_capture_dt_yyyymmdd TIMESTAMP,
    g_company_cd STRING,
    g_company_currency_cd STRING,
    g_dnsa_dt_yyyymmdd TIMESTAMP,
    g_expiration_dt_yyyymmdd TIMESTAMP,
    g_item_nbr STRING,
    g_location_cd STRING,
    g_location_status_cd STRING,
    g_lot_effective_dt_yyyymmdd TIMESTAMP,
    g_lot_nbr STRING,
    g_lot_status_cd STRING,
    g_plant_cd STRING,
    g_qty_allocated DECIMAL(10,2),
    g_qty_available DECIMAL(10,2),
    g_qty_financial_nettable DECIMAL(10,2),
    g_qty_in_transit DECIMAL(10,2),
    g_qty_mrp_nettable DECIMAL(10,2),
    g_qty_on_hand DECIMAL(10,2),
    g_qty_shippable DECIMAL(10,2),
    g_source_system_cd STRING,
    g_unit_cost_company_currency DECIMAL(10,2),
    plant_key STRING,
    plant_location_key STRING,
    prod_key STRING,
    prod_plant_key STRING,
    prod_plant_location_key STRING,
    prod_plant_lot_key STRING,
    prod_plant_lot_location_key STRING
);

/* 
Section: Schema Validation 
Description: Validate that the f_invntry_bal_dly_hist table schema matches the expected schema.
*/

-- Validate number of columns
SELECT
    COUNT(*) AS column_count
FROM
    INFORMATION_SCHEMA.COLUMNS
WHERE
    table_schema = 'purgo_playground'
    AND table_name = 'f_invntry_bal_dly_hist';

-- Expected: 29 columns
-- Uncomment the following line to assert the column count
-- SELECT CASE WHEN (SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS WHERE table_schema = 'purgo_playground' AND table_name = 'f_invntry_bal_dly_hist') = 29 THEN 'PASS' ELSE 'FAIL' END AS schema_column_count_validation;

-- Validate each column's data type
SELECT
    column_name,
    data_type
FROM
    INFORMATION_SCHEMA.COLUMNS
WHERE
    table_schema = 'purgo_playground'
    AND table_name = 'f_invntry_bal_dly_hist'
ORDER BY
    ordinal_position;

/* 
Section: Data Type Testing 
Description: Ensure all columns have the correct data types as per the specifications.
*/

-- Check for correct data types
SELECT
    column_name,
    data_type,
    CASE 
        WHEN column_name IN ('co_key', 'g_account', 'g_assignment_type', 'g_company_cd', 'g_company_currency_cd', 
                            'g_item_nbr', 'g_location_cd', 'g_location_status_cd', 'g_lot_nbr', 
                            'g_lot_status_cd', 'g_plant_cd', 'g_source_system_cd', 'plant_key', 
                            'plant_location_key', 'prod_key', 'prod_plant_key', 'prod_plant_location_key', 
                            'prod_plant_lot_key', 'prod_plant_lot_location_key') AND data_type = 'string' THEN 'PASS'
        WHEN column_name IN ('g_capture_dt_yyyymmdd', 'g_dnsa_dt_yyyymmdd', 'g_expiration_dt_yyyymmdd', 
                             'g_lot_effective_dt_yyyymmdd') AND data_type = 'timestamp' THEN 'PASS'
        WHEN column_name IN ('g_qty_allocated', 'g_qty_available', 'g_qty_financial_nettable', 
                             'g_qty_in_transit', 'g_qty_mrp_nettable', 'g_qty_on_hand', 'g_qty_shippable', 
                             'g_unit_cost_company_currency') AND data_type = 'decimal' THEN 'PASS'
        ELSE 'FAIL'
    END AS data_type_validation
FROM
    INFORMATION_SCHEMA.COLUMNS
WHERE
    table_schema = 'purgo_playground'
    AND table_name = 'f_invntry_bal_dly_hist';

-- Expected: All columns should have 'PASS' in data_type_validation

/* 
Section: Unit Tests for Individual Transformations 
Description: Test individual field transformations and derived logic.
*/

-- Test g_assignment_type is 'Account' when not represented
SELECT COUNT(*) AS incorrect_assignment_type
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_assignment_type <> 'Account';

-- Expected: 0

-- Test hardcoded g_company_cd as '1050'
SELECT COUNT(*) AS incorrect_company_cd
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_company_cd <> '1050';

-- Expected: 0

-- Test hardcoded g_source_system_cd as 'nav_ger'
SELECT COUNT(*) AS incorrect_source_system_cd
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_source_system_cd <> 'nav_ger';

-- Expected: 0

/* 
Section: Integration Tests for End-to-End Flows 
Description: Validate that end-to-end data transformations and loading are performing correctly.
*/

-- Validate that records from itemledgerentrieswopd are correctly transformed and loaded
SELECT COUNT(*) AS transformed_records
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_item_nbr IN ('ITEM001', 'ITEM002', 'ITEM003', 'ITEM015', 'ITEM016', 'ITEM017', 'ITEM018', 'ITEM019', 'ITEM020');

-- Expected: Number of test records inserted

-- Validate that all joined tables have corresponding records in f_invntry_bal_dly_hist
SELECT 
    COUNT(*) AS missing_records
FROM purgo_playground.itemledgerentrieswopd ile
LEFT JOIN purgo_playground.f_invntry_bal_dly_hist fibh
    ON TRIM(ile.item_no_) = TRIM(fibh.g_item_nbr)
    AND TRIM(ile.location_code) = TRIM(fibh.g_location_cd)
WHERE fibh.g_item_nbr IS NULL;

-- Expected: 0

/* 
Section: Data Quality Validation Tests 
Description: Ensure data quality rules such as uniqueness, completeness, and valid formats are enforced.
*/

-- Test uniqueness of g_item_nbr within the system
SELECT g_item_nbr, COUNT(*) AS record_count
FROM purgo_playground.f_invntry_bal_dly_hist
GROUP BY g_item_nbr
HAVING COUNT(*) > 1;

-- Expected: No records returned

-- Test completeness of g_account
SELECT COUNT(*) AS incomplete_g_account
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_account IS NULL OR g_account = '';

-- Expected: 0

-- Validate g_capture_dt_yyyymmdd is in YYYYMMDD format
SELECT COUNT(*) AS invalid_capture_dt
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE DATE_FORMAT(g_capture_dt_yyyymmdd, 'yyyyMMdd') IS NULL;

-- Expected: 0

-- Validate g_dnsa_dt_yyyymmdd is either valid date or default
SELECT COUNT(*) AS invalid_dnsa_dt
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE (g_dnsa_dt_yyyymmdd NOT BETWEEN '0001-01-01' AND '9999-12-31')
  AND g_dnsa_dt_yyyymmdd <> CAST('9999-12-31' AS TIMESTAMP);

-- Expected: 0

-- Validate g_expiration_dt_yyyymmdd is either valid date or default
SELECT COUNT(*) AS invalid_expiration_dt
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE (g_expiration_dt_yyyymmdd NOT BETWEEN '0001-01-01' AND '9999-12-31')
  AND g_expiration_dt_yyyymmdd <> CAST('9999-12-31' AS TIMESTAMP);

-- Expected: 0

/* 
Section: Delta Lake Operations Tests 
Description: Test Delta Lake specific operations like MERGE, UPDATE, DELETE.
*/

-- Test MERGE operation
MERGE INTO purgo_playground.f_invntry_bal_dly_hist AS target
USING (SELECT 'ITEMTEST1' AS g_item_nbr) AS source
ON target.g_item_nbr = source.g_item_nbr
WHEN MATCHED THEN UPDATE SET target.g_qty_available = 999.99
WHEN NOT MATCHED THEN INSERT (g_item_nbr, g_qty_available) VALUES (source.g_item_nbr, 999.99);

-- Validate MERGE
SELECT g_item_nbr, g_qty_available
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_item_nbr = 'ITEMTEST1';

-- Expected: g_qty_available = 999.99

-- Test DELETE operation
DELETE FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_item_nbr = 'ITEMTEST1';

-- Validate DELETE
SELECT COUNT(*) AS deleted_record
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_item_nbr = 'ITEMTEST1';

-- Expected: 0

/* 
Section: Window Functions and Analytics Tests 
Description: Validate the use of window functions and analytic features within the transformations.
*/

-- Example: Calculate running total of g_qty_available per g_location_cd
SELECT 
    g_location_cd,
    g_item_nbr,
    g_qty_available,
    SUM(g_qty_available) OVER (PARTITION BY g_location_cd ORDER BY g_item_nbr) AS running_total
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_location_cd = 'LOC001';

/* 
Section: Data Quality and NULL Handling Tests 
Description: Ensure NULLs are handled gracefully and default values are applied where necessary.
*/

-- Test NULL handling in g_qty_in_transit
SELECT COUNT(*) AS incorrect_qty_in_transit
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_qty_in_transit IS NOT NULL;

-- Expected: All records should have NULL in g_qty_in_transit

-- Validate default values for missing g_lot_nbr and g_serial_no_
SELECT COUNT(*) AS incorrect_defaults
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE (g_lot_nbr IS NULL OR g_lot_nbr = '') AND g_lot_nbr <> 'none';

-- Expected: 0

/* 
Section: Performance Tests 
Description: Assess the performance of data loading and transformations.
Note: Performance tests are typically conducted using benchmarking tools and may not be represented directly in SQL.
*/

-- Example: Measure time taken to load records
-- This is a placeholder as SQL alone cannot measure execution time
SELECT COUNT(*) FROM purgo_playground.f_invntry_bal_dly_hist;

/* 
Section: Cleanup Operations 
Description: Clean up test data to maintain a clean testing environment.
*/

-- Delete all test records created for testing
DELETE FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_item_nbr IN ('ITEMTEST1', 'ITEMHAPPY02', 'ITEMERR01', 'ITEMEXTDEC', 'ITEMMAP', 'ITEMMISS', 
                     'ITEMCOND', 'ITEMDEF', 'ITEMZERO', 'ITEMNEG', 'ITEM_NULL', 'ITEMDUP', 
                     'ITEMDUPLOC', 'ITEMHARDC', 'ITEMMAX', 'ITEMMIN', 'ITEMTIMESTAMP', 
                     'ITEMSPC', 'ITEMDUPLOC', 'ITEMEXTDEC', 'ITEMERR01');

/* 
Section: Additional Data Quality Rules Enforcement 
Description: Ensure all data quality rules are being enforced correctly.
*/

-- Enforce NOT NULL constraints where applicable
ALTER TABLE purgo_playground.f_invntry_bal_dly_hist
ALTER COLUMN g_account SET NOT NULL;

-- Validate that NOT NULL constraint is enforced
SELECT COUNT(*) AS not_null_constraint_failure
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_account IS NULL;

-- Expected: 0

/* 
Section: Foreign Key Relationships Validation 
Description: Ensure foreign key relationships are correctly established and enforced.
*/

-- Example: Validate foreign key relationship with sap_master_data
SELECT COUNT(*) AS foreign_key_violations
FROM purgo_playground.f_invntry_bal_dly_hist fibh
LEFT JOIN purgo_playground.sap_master_data smd
    ON fibh.g_item_nbr = smd.Item_No_
WHERE smd.Item_No_ IS NULL;

-- Expected: 0

/* 
Section: Snapshot Maintenance Schedule Verification 
Description: Ensure that snapshot maintenance adheres to the defined retention periods.
*/

-- Verify daily snapshots retention (100 days)
SELECT COUNT(*) AS daily_snapshots_retained
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_capture_dt_yyyymmdd >= DATEADD(day, -100, CURRENT_DATE());

-- Expected: 100

-- Verify weekly snapshots retention (3 years)
SELECT COUNT(*) AS weekly_snapshots_retained
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_capture_dt_yyyymmdd >= DATEADD(year, -3, CURRENT_DATE());

-- Expected: 156

-- Verify monthly snapshots retention (5 years)
SELECT COUNT(*) AS monthly_snapshots_retained
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_capture_dt_yyyymmdd >= DATEADD(year, -5, CURRENT_DATE());

-- Expected: 60

/* 
Section: Handling Missing or Invalid Data Gracefully 
Description: Ensure that the system handles missing or invalid data without failures.
*/

-- Validate that records with invalid Expiration_Date have default '99991231'
SELECT COUNT(*) AS default_expiration_applied
FROM purgo_playground.f_invntry_bal_dly_hist
WHERE g_expiration_dt_yyyymmdd = CAST('9999-12-31' AS TIMESTAMP)
  AND g_item_nbr = 'ITEM015';

-- Expected: 1

-- Validate error logs capture missing or invalid data
-- Placeholder as error logging mechanisms are external to SQL
SELECT * FROM purgo_playground.error_logs
WHERE Record_ID = 'ITEM015';

/* 
Section: Final Assertions 
Description: Summarize all test results to ensure all validations pass.
*/

-- Example: Overall test pass if all previous counts are zero
SELECT 
    (SELECT COUNT(*) FROM purgo_playground.f_invntry_bal_dly_hist WHERE g_assignment_type <> 'Account') AS incorrect_assignment_type,
    (SELECT COUNT(*) FROM purgo_playground.f_invntry_bal_dly_hist WHERE g_company_cd <> '1050') AS incorrect_company_cd,
    (SELECT COUNT(*) FROM purgo_playground.f_invntry_bal_dly_hist WHERE g_source_system_cd <> 'nav_ger') AS incorrect_source_system_cd,
    (SELECT COUNT(*) FROM purgo_playground.f_invntry_bal_dly_hist WHERE g_account IS NULL OR g_account = '') AS incomplete_g_account,
    (SELECT COUNT(*) FROM purgo_playground.f_invntry_bal_dly_hist WHERE g_item_nbr IN ('ITEM001', 'ITEM002', 'ITEM003') AND g_qty_available <> 100.00) AS incorrect_qty_available;

-- Expected: All columns should return 0
